# 0. Configuración inicial

#0.1. Verificar, guardar y crear

In [ ]:
# 1. Verificamos nuestra GPU T4 este activa y revisamos el espacio en /content, validen compañeros
!nvidia-smi
!df -h /content

# 2. Guardamos en nuestro Google Drive para guardar checkpoints y resultados de forma segura
from google.colab import drive
drive.mount('/content/drive')

# Creamos carpeta persistente en tu Drive universitario para respaldar los modelos
!mkdir -p /content/drive/MyDrive/Maestria_IA_CCT20/runs

#0.2. Clonar repositorio de GitHub y se procede a instalar librerías

In [ ]:
%cd /content
!git clone https://github.com/AlfonsoPe/camera-traps-yolo.git
%cd /content/camera-traps-yolo

# Crear la estructura de directorios local en /content
!mkdir -p /content/datasets/cct20/images /content/datasets/cct20/labels
!mkdir -p data/annotations configs src notebooks

# Instalar Ultralytics (YOLOv8/v9), PyCOCOTools, Grad-CAM y utilidades de descarga rápida como aria2
!pip install -q ultralytics pycocotools grad-cam pandas matplotlib seaborn tqdm
!apt-get update -qq && apt-get install -y -qq aria2

# 0.3. Descargamos los archivos JSON oficiales: Anotaciones y Partición eccv_split

In [ ]:
# 1. Limpiamos los archivos de 0 bytes del intento anterior
!rm -rf data/annotations/*
!mkdir -p data/annotations

# 2. Descargamos las particiones oficiales ECCV18 / CCT-20: train, cis_val, trans_val, cis_test, trans_test
!wget -q --show-progress -O data/annotations/eccv_18_annotations.tar.gz \
  https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz

# 3. Descargar las cajas delimitadoras bounding boxes y metadatos generales
!wget -q --show-progress -O data/annotations/caltech_bboxes_20200316.json.zip \
  https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/caltech_bboxes_20200316.json.zip

!wget -q --show-progress -O data/annotations/caltech_images_20210113.json.zip \
  https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/caltech_images_20210113.json.zip

# 4. Extraer los JSON
!tar -xzf data/annotations/eccv_18_annotations.tar.gz -C data/annotations/
!unzip -q -o data/annotations/caltech_bboxes_20200316.json.zip -d data/annotations/
!unzip -q -o data/annotations/caltech_images_20210113.json.zip -d data/annotations/

print("\nArchivos extraídos correctamente:")
!ls -lh data/annotations/
!ls -lh data/annotations/eccv_18_annotation_files/ 2>/dev/null || true

#0.4: Descargamos el paquete de imágenes CCT-20 de 6 GB

In [ ]:
#Descargamos y descomprimimos las imágenes oficiales del benchmark CCT-20 de 6 GB
!aria2c -x 16 -s 16 -d /content/datasets/cct20 -o eccv_18_all_images_sm.tar.gz \
  https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_all_images_sm.tar.gz

# Extraemos en el disco local rápido de Colab: /content/datasets/cct20
print("Extrayendo imágenes en /content/datasets/cct20/ ...")
!tar -xzf /content/datasets/cct20/eccv_18_all_images_sm.tar.gz -C /content/datasets/cct20/

# Eliminamos el archivo comprimido para liberar 6 GB de espacio en disco
!rm /content/datasets/cct20/eccv_18_all_images_sm.tar.gz
!ls -d /content/datasets/cct20/*

#0.5. Análisis Exploratorio y Conversión Automática a Formato YOLO


#Sin imagenes vacias, solo fue para probar nuestro código, no ejecutar en el proyecto

In [ ]:
# Procesamiento de CCT-20 a formato YOLO
import json
import os
import glob
import random
from pathlib import Path
from tqdm import tqdm

random.seed(42)

# 1. Aseguramos que eccv_18_annotations.tar.gz esté descargado y extraído usando gsutil que es un nativo en Colab
os.makedirs("data/annotations", exist_ok=True)
if not glob.glob("data/annotations/**/train_annotations.json", recursive=True):
    print("Descargando anotaciones oficiales de CCT-20 desde el bucket de Google Cloud de LILA...")
    !gsutil -m cp gs://public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz data/annotations/
    !tar -xzf data/annotations/eccv_18_annotations.tar.gz -C data/annotations/

# Localizamos automáticamente la carpeta donde quedaron los JSON: train_annotations.json, cis_val, trans_val y otros
train_json_matches = glob.glob("data/annotations/**/train_annotations.json", recursive=True)
if not train_json_matches:
    raise FileNotFoundError("No se encontró train_annotations.json. Verifica la descarga de eccv_18_annotations.tar.gz")

ann_base = os.path.dirname(train_json_matches[0])
print(f"✅ Carpeta de anotaciones localizada en: {ann_base}")

# Localizamos imágenes descargadas en la Celda anterior
img_source_dirs = glob.glob("/content/datasets/cct20/**/eccv_18_all_images_sm", recursive=True)
IMG_SRC_DIR = img_source_dirs[0] if img_source_dirs else "/content/datasets/cct20/eccv_18_all_images_sm"
print(f"✅ Directorio de imágenes fuente: {IMG_SRC_DIR}")

# 2. Leemos las categorías directamente desde train_annotations.json
with open(os.path.join(ann_base, "train_annotations.json"), "r") as f:
    train_meta = json.load(f)

# Mapeamos las especies animales a índices correlativos [0 .. N-1] y separar la clase 'empty'
raw_cats = sorted(train_meta["categories"], key=lambda x: x["id"])
species_cats = [c for c in raw_cats if c["name"].lower() not in ["empty", "vacío"]]
cat_id_to_yolo = {c["id"]: idx for idx, c in enumerate(species_cats)}
yolo_names = {idx: c["name"] for idx, c in enumerate(species_cats)}

print(f"✅ Total de clases de fauna con bounding box: {len(yolo_names)}")
print("Mapa de clases YOLO:", yolo_names)

# 3. Si además existe un archivo adicional de bboxes, indexarlo como respaldo; si no, usar los bboxes nativos del split
bboxes_master_by_img = {}
bbox_files = glob.glob("data/annotations/**/*bbox*.json", recursive=True)
if bbox_files:
    print(f"Cargando cajas adicionales desde: {bbox_files[0]}")
    with open(bbox_files[0], "r") as f:
        bdata = json.load(f)
    for ann in bdata.get("annotations", []):
        if ann.get("category_id") in cat_id_to_yolo and "bbox" in ann:
            bboxes_master_by_img.setdefault(ann["image_id"], []).append(
                (cat_id_to_yolo[ann["category_id"]], ann["bbox"])
            )

# 4. Función constructora de particiones YOLO : maneja bboxes nativos + imágenes vacías como fondo
YOLO_ROOT = Path("/content/datasets/cct20_yolo")

def build_yolo_split(split_json_name, split_folder_name, empty_ratio=0.15, is_train=True):
    json_path = os.path.join(ann_base, split_json_name)
    with open(json_path, "r") as f:
        split_data = json.load(f)

    img_out_dir = YOLO_ROOT / "images" / split_folder_name
    lbl_out_dir = YOLO_ROOT / "labels" / split_folder_name
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    empty_cat_ids = {c["id"] for c in split_data["categories"] if c["name"].lower() in ["empty", "vacío"]}

    # Agrupamos las anotaciones por image_id dentro de este split
    anns_by_img = {}
    for ann in split_data["annotations"]:
        anns_by_img.setdefault(ann["image_id"], []).append(ann)

    animal_imgs, empty_imgs = [], []
    for img in split_data["images"]:
        img_id = img["id"]
        img_anns = anns_by_img.get(img_id, [])

        # Verificar si es imagen vacía
        is_empty = any(a.get("category_id") in empty_cat_ids for a in img_anns)
        has_bbox = any(("bbox" in a and a.get("category_id") in cat_id_to_yolo) for a in img_anns) or (img_id in bboxes_master_by_img)

        if is_empty and not has_bbox:
            empty_imgs.append(img)
        elif has_bbox:
            animal_imgs.append(img)

    # Controlamos proporción de fotos vacías: 15% en train, 25% en val/test
    ratio = empty_ratio if is_train else 0.25
    if len(empty_imgs) > 0 and len(animal_imgs) > 0:
        n_keep = int(len(animal_imgs) * ratio)
        empty_imgs = random.sample(empty_imgs, min(n_keep, len(empty_imgs)))

    selected_imgs = animal_imgs + empty_imgs
    linked_count = 0

    for img in tqdm(selected_imgs, desc=f"Construyendo {split_folder_name}"):
        img_id = img["id"]
        fname = os.path.basename(img["file_name"])
        src_img_path = os.path.join(IMG_SRC_DIR, fname)
        if not os.path.exists(src_img_path):
            continue

        # Creamos symlink en /content/datasets/cct20_yolo/images/<split>
        dst_img_path = img_out_dir / fname
        if not dst_img_path.exists():
            os.symlink(src_img_path, dst_img_path)

        w_img = img.get("width", 1024)
        h_img = img.get("height", 747)

        # Extraemos bounding boxes del propio split_data o del archivo maestro
        boxes_to_write = []
        for a in anns_by_img.get(img_id, []):
            if "bbox" in a and a.get("category_id") in cat_id_to_yolo:
                boxes_to_write.append((cat_id_to_yolo[a["category_id"]], a["bbox"]))
        if not boxes_to_write and img_id in bboxes_master_by_img:
            boxes_to_write = bboxes_master_by_img[img_id]

        # Convertimos [x_min, y_min, w, h] -> YOLO [cls, xc, yc, wn, hn]
        lines = []
        for cls_idx, bbox in boxes_to_write:
            x_min, y_min, bw, bh = bbox
            xc = max(0.0, min(1.0, (x_min + bw / 2.0) / w_img))
            yc = max(0.0, min(1.0, (y_min + bh / 2.0) / h_img))
            wn = max(0.0, min(1.0, bw / w_img))
            hn = max(0.0, min(1.0, bh / h_img))
            lines.append(f"{cls_idx} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")

        lbl_path = lbl_out_dir / f"{Path(fname).stem}.txt"
        with open(lbl_path, "w") as lf:
            lf.write("\n".join(lines))
        linked_count += 1

    print(f"📊 [{split_folder_name}] Imágenes listas: {linked_count} (Con fauna: {len(animal_imgs)} | Vacías de fondo: {len(empty_imgs)})")

# 5. Ejecutamos para todas las particiones oficiales de CCT-20
build_yolo_split("train_annotations.json", "train", empty_ratio=0.15, is_train=True)
build_yolo_split("cis_val_annotations.json", "cis_val", is_train=False)
build_yolo_split("trans_val_annotations.json", "trans_val", is_train=False)
build_yolo_split("cis_test_annotations.json", "cis_test", is_train=False)
build_yolo_split("trans_test_annotations.json", "trans_test", is_train=False)

# 6. Creamos configs/cct20.yaml para YOLOv8
os.makedirs("configs", exist_ok=True)
yaml_content = f"""path: /content/datasets/cct20_yolo
train: images/train
val: images/cis_val
test: images/trans_val

names:
"""
for idx, name in yolo_names.items():
    yaml_content += f"  {idx}: {name}\n"

with open("configs/cct20.yaml", "w") as f:
    f.write(yaml_content)

print("\n✅ ¡Dataset CCT-20 en formato YOLO y archivo configs/cct20.yaml creados con éxito!")

#Incluyendo imagenes vacias, de acuerdo al encunciado del documento

In [ ]:
# Procesamiento de CCT-20 a formato YOLO con manejo explícito de imágenes vacías
import json
import os
import glob
import random
from pathlib import Path
from tqdm import tqdm

random.seed(42)

# 1. Localizamos la carpeta de anotaciones de eccv_18
train_json_matches = glob.glob("data/annotations/**/train_annotations.json", recursive=True)
if not train_json_matches:
    raise FileNotFoundError("No se encontró train_annotations.json. Asegúrate de haber ejecutado la Celda 3.")

ann_base = os.path.dirname(train_json_matches[0])
print(f"✅ Carpeta de anotaciones localizada en: {ann_base}")

# Localizamos imágenes descargadas en la Celda 4
img_source_dirs = glob.glob("/content/datasets/cct20/**/eccv_18_all_images_sm", recursive=True)
IMG_SRC_DIR = img_source_dirs[0] if img_source_dirs else "/content/datasets/cct20/eccv_18_all_images_sm"
print(f"✅ Directorio de imágenes fuente: {IMG_SRC_DIR}")

# 2. Leemos categorías desde train_annotations.json
with open(os.path.join(ann_base, "train_annotations.json"), "r") as f:
    train_meta = json.load(f)

raw_cats = sorted(train_meta["categories"], key=lambda x: x["id"])
species_cats = [c for c in raw_cats if c["name"].lower() not in ["empty", "vacío"]]
cat_id_to_yolo = {c["id"]: idx for idx, c in enumerate(species_cats)}
yolo_names = {idx: c["name"] for idx, c in enumerate(species_cats)}

print(f"✅ Total de clases de fauna con bounding box: {len(yolo_names)}")

# 3. Función constructora de particiones YOLO con muestreo explícito de imágenes vacías
YOLO_ROOT = Path("/content/datasets/cct20_yolo")

def build_yolo_split_fixed(split_json_name, split_folder_name, empty_ratio=0.15, is_train=True):
    json_path = os.path.join(ann_base, split_json_name)
    with open(json_path, "r") as f:
        split_data = json.load(f)

    img_out_dir = YOLO_ROOT / "images" / split_folder_name
    lbl_out_dir = YOLO_ROOT / "labels" / split_folder_name
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    # Agrupamos anotaciones por image_id
    anns_by_img = {}
    for ann in split_data["annotations"]:
        anns_by_img.setdefault(ann["image_id"], []).append(ann)

    animal_imgs, empty_imgs = [], []
    for img in split_data["images"]:
        img_id = img["id"]
        img_anns = anns_by_img.get(img_id, [])

        # Filtramos qué anotaciones pertenecen a clases de fauna válidas
        valid_boxes = [a for a in img_anns if "bbox" in a and a.get("category_id") in cat_id_to_yolo]

        if len(valid_boxes) > 0:
            animal_imgs.append((img, valid_boxes))
        else:
            empty_imgs.append((img, []))

    # Controlamos proporción de fotos vacías: 15% en train, 25% en val/test
    ratio = empty_ratio if is_train else 0.25
    if len(empty_imgs) > 0 and len(animal_imgs) > 0:
        n_keep = int(len(animal_imgs) * ratio)
        empty_imgs = random.sample(empty_imgs, min(n_keep, len(empty_imgs)))

    selected_data = animal_imgs + empty_imgs
    random.shuffle(selected_data)
    linked_count = 0

    for img, valid_boxes in tqdm(selected_data, desc=f"Construyendo {split_folder_name}"):
        fname = os.path.basename(img["file_name"])
        src_img_path = os.path.join(IMG_SRC_DIR, fname)
        if not os.path.exists(src_img_path):
            continue

        # Creamos symlink en /content/datasets/cct20_yolo/images/<split>
        dst_img_path = img_out_dir / fname
        if not dst_img_path.exists():
            os.symlink(src_img_path, dst_img_path)

        w_img = img.get("width", 1024)
        h_img = img.get("height", 747)

        # Convertimos cajas a formato YOLO [cls, xc, yc, wn, hn]
        lines = []
        for a in valid_boxes:
            cls_idx = cat_id_to_yolo[a["category_id"]]
            x_min, y_min, bw, bh = a["bbox"]
            xc = max(0.0, min(1.0, (x_min + bw / 2.0) / w_img))
            yc = max(0.0, min(1.0, (y_min + bh / 2.0) / h_img))
            wn = max(0.0, min(1.0, bw / w_img))
            hn = max(0.0, min(1.0, bh / h_img))
            lines.append(f"{cls_idx} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")

        # Si es imagen vacía, se crea un archivo .txt vacío (clase background)
        lbl_path = lbl_out_dir / f"{Path(fname).stem}.txt"
        with open(lbl_path, "w") as lf:
            lf.write("\n".join(lines))
        linked_count += 1

    print(f"📊 [{split_folder_name}] Imágenes listas: {linked_count} (Con fauna: {len(animal_imgs)} | Vacías de fondo muestreadas: {len(empty_imgs)})")

# 4. Ejecutamos para todas las particiones oficiales de CCT-20
build_yolo_split_fixed("train_annotations.json", "train", empty_ratio=0.15, is_train=True)
build_yolo_split_fixed("cis_val_annotations.json", "cis_val", is_train=False)
build_yolo_split_fixed("trans_val_annotations.json", "trans_val", is_train=False)
build_yolo_split_fixed("cis_test_annotations.json", "cis_test", is_train=False)
build_yolo_split_fixed("trans_test_annotations.json", "trans_test", is_train=False)

# 5. Creamos configs/cct20.yaml
os.makedirs("configs", exist_ok=True)
yaml_content = f"""path: /content/datasets/cct20_yolo
train: images/train
val: images/cis_val
test: images/trans_val

names:
"""
for idx, name in yolo_names.items():
    yaml_content += f"  {idx}: {name}\n"

with open("configs/cct20.yaml", "w") as f:
    f.write(yaml_content)

print("\n ¡Dataset actualizado con imágenes vacías en train y configs/cct20.yaml listo!")

#**Entregable 1** — Entrenamiento del Modelo YOLOv8

In [16]:

from ultralytics import YOLO
import os

# 1. Cargamos el modelo base preentrenado: YOLOv8s
model = YOLO('yolov8s.pt')

# 2. Configuramos ruta de respaldo persistente en Google Drive
DRIVE_RUNS_DIR = '/content/drive/MyDrive/Maestria_IA_CCT20/runs'
os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)

# 3. Ejecutamos entrenamiento con hiperparámetros optimizados para cámaras trampa y GPU T4
results = model.train(
    data='configs/cct20.yaml',
    epochs=25,             # número de épocas
    imgsz=640,             # Resolución de entrada estándar y óptima para T4
    batch=32,              # Tamaño de lote eficiente para 16 GB VRAM con AMP activado
    amp=True,              # Precisión mixta automática (acelera el entrenamiento)
    workers=4,             # Hilos de carga de datos optimizados para Colab
    mosaic=1.0,            # Aumento Mosaic para combinar múltiples fondos y reducir sobreajuste al fondo
    mixup=0.15,            # Aumento MixUp para robustez ante camuflaje
    copy_paste=0.1,        # Copiar animales a distintos fondos
    project=DRIVE_RUNS_DIR, # Guardamos directamente en Google Drive
    name='cct20_yolov8s',
    seed=42,
    save=True,
    plots=True
)

print("\n ¡Entrenamiento completado y respaldado en Google Drive!")

#Revisamos el estado real, despues de que ejecutamos las epocas


In [18]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/Maestria_IA_CCT20/runs/cct20_yolov8s-7/weights/best.pt')

print("=== CIS_VAL (ubicaciones vistas) ===")
metrics_cis = model.val(data='configs/cct20.yaml', split='val')

print("\n=== TRANS_VAL (ubicaciones NUEVAS) ===")
metrics_trans = model.val(data='configs/cct20.yaml', split='test')

#**Entregable 2** — Efecto del desbalance de clases
#**Entregable 3** — Parte 1 - Brecha CIS vs TRANS

In [19]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# 1. Contamos las instancias reales por clase en el conjunto de ENTRENAMIENTO
train_label_files = glob.glob('/content/datasets/cct20_yolo/labels/train/*.txt')
train_counts = Counter()
empty_train_files = 0

for lf in train_label_files:
    with open(lf, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]
        if not lines:
            empty_train_files += 1
        for line in lines:
            cls_id = int(line.split()[0])
            train_counts[cls_id] += 1

print(f"Total de imágenes vacías (background) verificadas en train: {empty_train_files}")

# 2. Extraemos métricas por clase desde metrics_cis y metrics_trans (ya calculados en la celda anterior)
names = metrics_cis.names  # Diccionario {0: 'opossum', ...}

# Mapeamos los resultados de CIS_VAL (ubicaciones vistas)
cis_map50 = dict(zip(metrics_cis.ap_class_index, metrics_cis.box.ap50))
cis_map95 = dict(zip(metrics_cis.ap_class_index, metrics_cis.box.ap))

# Mapeamos los resultados de TRANS_VAL (ubicaciones nuevas)
trans_map50 = dict(zip(metrics_trans.ap_class_index, metrics_trans.box.ap50))
trans_map95 = dict(zip(metrics_trans.ap_class_index, metrics_trans.box.ap))

# 3. Construimos Tabla Maestra comparativa
rows = []
for cls_id, sp_name in names.items():
    n_train = train_counts.get(cls_id, 0)
    c50 = cis_map50.get(cls_id, np.nan)
    c95 = cis_map95.get(cls_id, np.nan)
    t50 = trans_map50.get(cls_id, np.nan)
    t95 = trans_map95.get(cls_id, np.nan)
    gap50 = (c50 - t50) if (not np.isnan(c50) and not np.isnan(t50)) else np.nan

    rows.append({
        'Class_ID': cls_id,
        'Especie': sp_name,
        'Instancias_Train': n_train,
        'CIS_mAP@0.5': round(c50, 3) if not np.isnan(c50) else None,
        'CIS_mAP@0.5:0.95': round(c95, 3) if not np.isnan(c95) else None,
        'TRANS_mAP@0.5': round(t50, 3) if not np.isnan(t50) else None,
        'TRANS_mAP@0.5:0.95': round(t95, 3) if not np.isnan(t95) else None,
        'Brecha_mAP@0.5 (CIS - TRANS)': round(gap50, 3) if not np.isnan(gap50) else None
    })

df_entregable2_3 = pd.DataFrame(rows).sort_values(by='Instancias_Train', ascending=False)
print("\n📊 TABLA OFICIAL PARA ENTREGABLES 2 Y 3 , Ordenados de especie más común a más rara en Train:")
display(df_entregable2_3)

# Guardamos CSV en Drive para nuestro informe d ela pregunta 3
df_entregable2_3.to_csv('/content/drive/MyDrive/Maestria_IA_CCT20/tabla_entregables_2_y_3.csv', index=False)

# 4. GRÁFICO 1 del Entregable 2: Desbalance en Train vs mAP@0.5
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_valid_cis = df_entregable2_3.dropna(subset=['CIS_mAP@0.5'])
sns.barplot(
    data=df_entregable2_3, x='Instancias_Train', y='Especie', ax=axes[0], palette='viridis'
)
axes[0].set_xscale('log')
axes[0].set_title('Distribución de Cola Larga en Entrenamiento (Escala Log)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('N.° de Bounding Boxes en Train (log)')
axes[0].set_ylabel('Especie')
axes[0].grid(True, axis='x', linestyle='--', alpha=0.5)

# Scatter plot: N_train vs mAP@0.5 en CIS y TRANS
axes[1].scatter(df_valid_cis['Instancias_Train'], df_valid_cis['CIS_mAP@0.5'],
                color='royalblue', s=90, label='CIS (Cámaras vistas)', alpha=0.85)
df_valid_trans = df_entregable2_3.dropna(subset=['TRANS_mAP@0.5'])
axes[1].scatter(df_valid_trans['Instancias_Train'], df_valid_trans['TRANS_mAP@0.5'],
                color='crimson', s=90, marker='^', label='TRANS (Cámaras nuevas)', alpha=0.85)

for _, row in df_valid_cis.iterrows():
    axes[1].annotate(row['Especie'], (row['Instancias_Train'], row['CIS_mAP@0.5']),
                     xytext=(5, 4), textcoords='offset points', fontsize=9)

axes[1].set_xscale('log')
axes[1].set_ylim(-0.05, 1.08)
axes[1].set_title('Efecto del Desbalance de Clases sobre mAP@0.5', fontsize=12, fontweight='bold')
axes[1].set_xlabel('N.° de Instancias en Entrenamiento (Escala Log)')
axes[1].set_ylabel('mAP@0.5')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Maestria_IA_CCT20/fig_entregable2_desbalance.png', dpi=300)
plt.show()

#**Entregable 3** — Parte 2 - Visualización de Grad-CAM

In [20]:
# Acá compañeros hacemos el Análisis de Explicabilidad con Grad-CAM sobre el Backbone de YOLOv8
import torch
import cv2
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Cargar el mejor modelo entrenado
BEST_WEIGHTS = '/content/drive/MyDrive/Maestria_IA_CCT20/runs/cct20_yolov8s-7/weights/best.pt'
yolo_model = YOLO(BEST_WEIGHTS)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pytorch_model = yolo_model.model.to(device).eval()

# Activar gradientes en los parámetros del backbone para poder calcular Grad-CAM
for param in pytorch_model.parameters():
    param.requires_grad_(True)

# 2. Definir Hook sobre la última capa del Backbone de YOLOv8 con Capa 9: SPPF
target_layer = pytorch_model.model[9]
activations = {}
gradients = {}

def forward_hook(module, input, output):
    activations['value'] = output

def backward_hook(module, grad_input, grad_output):
    gradients['value'] = grad_output[0]

handle_f = target_layer.register_forward_hook(forward_hook)
handle_b = target_layer.register_full_backward_hook(backward_hook)

def compute_yolo_gradcam(img_path, imgsz=640):
    # Leer y preprocesar imagen
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = img_rgb.shape[:2]
    img_resized = cv2.resize(img_rgb, (imgsz, imgsz))

    tensor_img = torch.from_numpy(img_resized).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
    tensor_img.requires_grad_(True)

    pytorch_model.zero_grad()
    preds = pytorch_model(tensor_img)

    # preds[0] en YOLOv8 eval tiene forma [1, 4 + num_classes, 8400]
    pred_tensor = preds[0] if isinstance(preds, (list, tuple)) else preds
    class_scores = pred_tensor[0, 4:, :]  # [15, 8400]

    # Tomar el score máximo de clasificación como objetivo diferenciable para Grad-CAM
    max_score = class_scores.max()
    max_score.backward()

    grads = gradients['value'][0]       # [C, H, W]
    fmaps = activations['value'][0]     # [C, H, W]

    # Ponderación global de gradientes por canal (ecuación estándar de Grad-CAM)
    weights = grads.mean(dim=(1, 2), keepdim=True)
    cam = torch.relu((weights * fmaps).sum(dim=0)).detach().cpu().numpy()

    # Normalizar mapa de calor entre 0 y 1 y redimensionar al tamaño original
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam_resized = cv2.resize(cam, (w_orig, h_orig))

    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = np.uint8(0.55 * img_rgb + 0.45 * heatmap)

    # Obtener predicción final con cajas dibujadas por Ultralytics
    res = yolo_model.predict(img_path, imgsz=imgsz, verbose=False)[0]
    pred_img = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)

    return img_rgb, overlay, pred_img, res

# 3. Seleccionamos las muestras con objetos reales de CIS_VAL: vistas y TRANS_VAL:nuevas
def get_sample_images_with_animals(split_name, k=2):
    lbl_files = sorted(glob.glob(f'/content/datasets/cct20_yolo/labels/{split_name}/*.txt'))
    selected = []
    for lf in lbl_files:
        with open(lf) as f:
            lines = [l.strip() for l in f if l.strip()]
        if lines:
            stem = os.path.splitext(os.path.basename(lf))[0]
            img_matches = glob.glob(f'/content/datasets/cct20_yolo/images/{split_name}/{stem}.*')
            if img_matches:
                selected.append((img_matches[0], lines))
        if len(selected) >= k:
            break
    return selected

cis_samples = get_sample_images_with_animals('cis_val', k=2)
trans_samples = get_sample_images_with_animals('trans_val', k=2)
all_samples = [('CIS_VAL (Cámara Vista)', s[0], s[1]) for s in cis_samples] + \
              [('TRANS_VAL (Cámara Nueva)', s[0], s[1]) for s in trans_samples]

# 4. Graficamos comparativa Grad-CAM (Original + GT vs Mapa Grad-CAM Backbone vs Predicción YOLO)
fig, axes = plt.subplots(len(all_samples), 3, figsize=(16, 4.2 * len(all_samples)))
names_dict = yolo_model.names

for idx, (split_label, img_path, gt_lines) in enumerate(all_samples):
    orig_rgb, cam_overlay, pred_rgb, res = compute_yolo_gradcam(img_path)
    gt_classes = [names_dict[int(l.split()[0])] for l in gt_lines]

    # Columna 1: Imagen Original + Etiqueta Real
    axes[idx, 0].imshow(orig_rgb)
    axes[idx, 0].set_title(f"{split_label}\nGT Real: {', '.join(gt_classes)}", fontsize=11, fontweight='bold')
    axes[idx, 0].axis('off')

    # Columna 2: Mapa de Activación Grad-CAM (Capa 9 SPPF del Backbone)
    axes[idx, 1].imshow(cam_overlay)
    axes[idx, 1].set_title("Grad-CAM Backbone (Capa SPPF)\n(Zonas rojas = mayor atención de la red)", fontsize=11)
    axes[idx, 1].axis('off')

    # Columna 3: Detección Final de YOLOv8
    axes[idx, 2].imshow(pred_rgb)
    n_dets = len(res.boxes)
    axes[idx, 2].set_title(f"Predicción YOLOv8 ({n_dets} detecciones)", fontsize=11)
    axes[idx, 2].axis('off')

# Liberar hooks al finalizar
handle_f.remove()
handle_b.remove()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Maestria_IA_CCT20/fig_entregable3_gradcam.png', dpi=300)
plt.show()
print("Figura Grad-CAM guardada en /content/drive/MyDrive/Maestria_IA_CCT20/fig_entregable3_gradcam.png")

**Entregable 4**: Análisis de robustez ante oclusión y tamaño del objeto. - Oclusión y tamaño de objeto

In [21]:
import os
import glob
import shutil
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO

BEST_WEIGHTS = '/content/drive/MyDrive/Maestria_IA_CCT20/runs/cct20_yolov8s-7/weights/best.pt'
model = YOLO(BEST_WEIGHTS)

# Usamos la partición de evaluación para medir limpiamente el efecto de tamaño y oclusión
SRC_IMG_DIR = Path('/content/datasets/cct20_yolo/images/cis_val')
SRC_LBL_DIR = Path('/content/datasets/cct20_yolo/labels/cis_val')
ROBUST_ROOT = Path('/content/datasets/cct20_robustness')

if ROBUST_ROOT.exists():
    shutil.rmtree(ROBUST_ROOT)

# 1. Calculamos el área relativa e índice estimado de oclusión para cada imagen con fauna
records = []
lbl_files = sorted(glob.glob(str(SRC_LBL_DIR / '*.txt')))

for lf in tqdm(lbl_files, desc="Calculando métricas de Tamaño y Oclusión"):
    with open(lf, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    if not lines:
        continue  # esto compañeros es para ignorar imágenes vacías para agrupar estrictamente por propiedades del objeto

    stem = Path(lf).stem
    img_matches = list(SRC_IMG_DIR.glob(f"{stem}.*"))
    if not img_matches:
        continue
    img_path = img_matches[0]

    # Calcular área relativa promedio de las cajas en la imagen e indicadores de oclusión
    areas = []
    border_touch = 0
    aspect_distortion = 0

    img_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    h_img, w_img = img_gray.shape[:2] if img_gray is not None else (747, 1024)
    edge_density_list = []

    for l in lines:
        parts = [float(x) for x in l.split()]
        _, xc, yc, wn, hn = parts
        area = wn * hn
        areas.append(area)

        # 1) Truncamiento en bordes de la cámara
        xmin, xmax = xc - wn/2, xc + wn/2
        ymin, ymax = yc - hn/2, yc + hn/2
        if xmin <= 0.015 or xmax >= 0.985 or ymin <= 0.015 or ymax >= 0.985:
            border_touch += 1

        # 2) Relación de aspecto extrema (cuerpo parcialmente tapado por troncos/vegetación)
        ar = max(wn / (hn + 1e-6), hn / (wn + 1e-6))
        if ar > 2.2:
            aspect_distortion += (ar - 2.2)

        # 3) Densidad de bordes internos de alta frecuencia (ramas/hojas cruzando el bounding box)
        if img_gray is not None:
            x1, x2 = max(0, int(xmin * w_img)), min(w_img, int(xmax * w_img))
            y1, y2 = max(0, int(ymin * h_img)), min(h_img, int(ymax * h_img))
            crop = img_gray[y1:y2, x1:x2]
            if crop.size > 100:
                edges = cv2.Canny(crop, 80, 160)
                edge_density_list.append(np.mean(edges > 0))
            else:
                edge_density_list.append(0.15)

    mean_area = np.mean(areas)
    mean_edge = np.mean(edge_density_list) if edge_density_list else 0.1

    # Clasificación por Tamaño Relativo del Bounding Box
    if mean_area < 0.02:
        size_group = '1_Pequeno (<2%)'
    elif mean_area < 0.10:
        size_group = '2_Mediano (2%-10%)'
    else:
        size_group = '3_Grande (>=10%)'

    # Score compuesto de oclusión: Truncamiento + Distorsión + Oclusión por vegetación/bordes
    occ_score = (border_touch * 0.45) + (aspect_distortion * 0.25) + (mean_edge * 1.5)

    records.append({
        'stem': stem,
        'img_path': img_path,
        'lbl_path': Path(lf),
        'mean_area': mean_area,
        'size_group': size_group,
        'occ_score': occ_score
    })

df_meta = pd.DataFrame(records)

# Clasificar Nivel de Oclusión en 3 terciles: Bajo, Medio, Alto
q33, q66 = df_meta['occ_score'].quantile([0.333, 0.666]).values
def assign_occ_level(score):
    if score <= q33:
        return '1_Oclusion_Baja'
    elif score <= q66:
        return '2_Oclusion_Media'
    else:
        return '3_Oclusion_Alta'

df_meta['occ_group'] = df_meta['occ_score'].apply(assign_occ_level)

# 2. Función auxiliar para crear subconjunto YOLO y evaluar con model.val()
with open('configs/cct20.yaml', 'r') as f:
    base_yaml = yaml.safe_load(f)
names_map = base_yaml['names']

def evaluate_subset(subset_df, subset_name):
    img_dir = ROBUST_ROOT / 'images' / subset_name
    lbl_dir = ROBUST_ROOT / 'labels' / subset_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for _, row in subset_df.iterrows():
        dst_img = img_dir / row['img_path'].name
        dst_lbl = lbl_dir / row['lbl_path'].name
        if not dst_img.exists():
            os.symlink(row['img_path'], dst_img)
        if not dst_lbl.exists():
            os.symlink(row['lbl_path'], dst_lbl)

    yaml_path = ROBUST_ROOT / f"{subset_name}.yaml"
    yaml_data = {
        'path': str(ROBUST_ROOT),
        'train': 'images/train',
        'val': f"images/{subset_name}",
        'names': names_map
    }
    with open(yaml_path, 'w') as yf:
        yaml.dump(yaml_data, yf)

    metrics = model.val(data=str(yaml_path), split='val', verbose=False, plots=False)
    return {
        'Grupo': subset_name,
        'N_Imagenes': len(subset_df),
        'Precision': round(float(metrics.box.mp), 3),
        'Recall': round(float(metrics.box.mr), 3),
        'mAP@0.5': round(float(metrics.box.map50), 3),
        'mAP@0.5:0.95': round(float(metrics.box.map), 3)
    }

# 3. Evaluar los 3 grupos de Tamaño y los 3 grupos de Oclusión
print("\n Evaluando desempeño por TAMAÑO RELATIVO DEL BOUNDING BOX...")
size_results = []
for grp in sorted(df_meta['size_group'].unique()):
    sub_df = df_meta[df_meta['size_group'] == grp]
    size_results.append(evaluate_subset(sub_df, grp))
df_size_res = pd.DataFrame(size_results)
max_map_size = df_size_res['mAP@0.5'].max()
df_size_res['Caida_mAP@0.5_vs_Max'] = (df_size_res['mAP@0.5'] - max_map_size).round(3)

print("\n Evaluando desempeño por NIVEL ESTIMADO DE OCLUSIÓN...")
occ_results = []
for grp in sorted(df_meta['occ_group'].unique()):
    sub_df = df_meta[df_meta['occ_group'] == grp]
    occ_results.append(evaluate_subset(sub_df, grp))
df_occ_res = pd.DataFrame(occ_results)
max_map_occ = df_occ_res['mAP@0.5'].max()
df_occ_res['Caida_mAP@0.5_vs_Baja'] = (df_occ_res['mAP@0.5'] - max_map_occ).round(3)

print("\n TABLA 4A — ROBUSTEZ POR TAMAÑO RELATIVO DEL BOUNDING BOX:")
display(df_size_res)

print("\n TABLA 4B — ROBUSTEZ POR NIVEL ESTIMADO DE OCLUSIÓN:")
display(df_occ_res)

# Guardar tablas en Drive
df_size_res.to_csv('/content/drive/MyDrive/Maestria_IA_CCT20/tabla_entregable4_tamano.csv', index=False)
df_occ_res.to_csv('/content/drive/MyDrive/Maestria_IA_CCT20/tabla_entregable4_oclusion.csv', index=False)

# 4. Graficar caída de desempeño para el informe
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

x1 = np.arange(len(df_size_res))
w = 0.35
axes[0].bar(x1 - w/2, df_size_res['mAP@0.5'], width=w, label='mAP@0.5', color='steelblue')
axes[0].bar(x1 + w/2, df_size_res['mAP@0.5:0.95'], width=w, label='mAP@0.5:0.95', color='darkorange')
axes[0].set_xticks(x1)
axes[0].set_xticklabels(['Pequeño (<2%)', 'Mediano (2%-10%)', 'Grande (>=10%)'])
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Impacto del Tamaño del Bounding Box en el Desempeño', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Métrica (mAP)')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

x2 = np.arange(len(df_occ_res))
axes[1].bar(x2 - w/2, df_occ_res['mAP@0.5'], width=w, label='mAP@0.5', color='seagreen')
axes[1].bar(x2 + w/2, df_occ_res['mAP@0.5:0.95'], width=w, label='mAP@0.5:0.95', color='indianred')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(['Oclusión Baja', 'Oclusión Media', 'Oclusión Alta'])
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Impacto del Nivel Estimado de Oclusión en el Desempeño', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Métrica (mAP)')
axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Maestria_IA_CCT20/fig_entregable4_robustez.png', dpi=300)
plt.show()

**Entregable 5**: Módulo de Posprocesamiento de Monitoreo Ecológico, en camara no vista

In [22]:
import json, os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

BEST_WEIGHTS = '/content/drive/MyDrive/Maestria_IA_CCT20/runs/cct20_yolov8s-7/weights/best.pt'
model = YOLO(BEST_WEIGHTS)
names_map = model.names

# 1. Cargamos metadatos temporales y de ubicación desde trans_val_annotations.json (cámaras NO vistas)
ann_files = glob.glob("data/annotations/**/trans_val_annotations.json", recursive=True)
with open(ann_files[0], 'r') as f:
    trans_data = json.load(f)

trans_img_dir = '/content/datasets/cct20_yolo/images/trans_val'
available_files = set(os.listdir(trans_img_dir))

# Filtramos imágenes presentes en trans_val y extraer fecha/hora (date_captured) y cámara (location)
img_meta_list = []
for img in trans_data['images']:
    fname = os.path.basename(img['file_name'])
    if fname in available_files:
        img_meta_list.append({
            'image_id': img['id'],
            'file_name': fname,
            'img_path': os.path.join(trans_img_dir, fname),
            'location': str(img.get('location', 'cam_unknown')),
            'date_captured': img.get('date_captured', None),
            'seq_id': img.get('seq_id', None)
        })

df_trans_meta = pd.DataFrame(img_meta_list)
df_trans_meta['datetime'] = pd.to_datetime(df_trans_meta['date_captured'], errors='coerce')

# Seleccionamos la cámara no vista con mayor número de fotogramas para la demostración
top_cam = df_trans_meta['location'].value_counts().index[0]
df_cam = df_trans_meta[df_trans_meta['location'] == top_cam].copy()
df_cam = df_cam.sort_values(by=['datetime', 'file_name']).reset_index(drop=True)

# Si alguna foto no tiene timestamp válido, asignar secuencia sintética ordenada cada 12 minutos
if df_cam['datetime'].isna().all():
    df_cam['datetime'] = pd.date_range(start='2018-05-01 00:00:00', periods=len(df_cam), freq='12min')
else:
    df_cam['datetime'] = df_cam['datetime'].ffill().bfill()

print(f"Cámara no vista seleccionada para monitoreo ecológico: Ubicación #{top_cam} ({len(df_cam)} fotogramas)")

# 2. Ejecutamos inferencia YOLO por fotograma sobre toda la secuencia de la cámara
frame_Logs = []
for idx, row in df_cam.iterrows():
    res = model.predict(row['img_path'], imgsz=640, conf=0.35, verbose=False)[0]
    boxes = res.boxes
    if len(boxes) == 0:
        frame_Logs.append({
            'frame_idx': idx, 'file_name': row['file_name'], 'location': row['location'],
            'datetime': row['datetime'], 'is_empty': True, 'species': 'empty',
            'n_individuals': 0, 'max_conf': 0.0
        })
    else:
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        # Regla 1: Especie dominante en el fotograma y conteo de individuos por cuadro
        top_idx = np.argmax(confs)
        dom_species = names_map[cls_ids[top_idx]]
        frame_Logs.append({
            'frame_idx': idx, 'file_name': row['file_name'], 'location': row['location'],
            'datetime': row['datetime'], 'is_empty': False, 'species': dom_species,
            'n_individuals': len(boxes), 'max_conf': float(confs[top_idx])
        })

df_frames = pd.DataFrame(frame_Logs)

# 3. Regla 2 de Monitoreo Ecológico: Agrupación en Eventos Independientes: ventana Delta_t = 30 min
DELTA_MIN = 30
df_animals = df_frames[~df_frames['is_empty']].copy().sort_values('datetime')

eventos = []
for sp, grp in df_animals.groupby('species'):
    last_time = None
    event_id = 0
    for _, r in grp.iterrows():
        if last_time is None or (r['datetime'] - last_time).total_seconds() / 60.0 > DELTA_MIN:
            event_id += 1
        last_time = r['datetime']
        eventos.append({**r.to_dict(), 'independent_event_id': f"{sp}_ev{event_id}"})

df_eventos = pd.DataFrame(eventos)

# Asignamos Franja Horaria Ecológica: Madrugada, Mañana, Tarde, Noche
def get_period(dt):
    hr = dt.hour
    if 0 <= hr < 6: return '1_Madrugada (00-06h)'
    elif 6 <= hr < 12: return '2_Mañana (06-12h)'
    elif 12 <= hr < 18: return '3_Tarde (12-18h)'
    else: return '4_Noche (18-24h)'

df_frames['Periodo_Dia'] = df_frames['datetime'].apply(get_period)

# 4. Generamos Reporte 1: Resumen General de la Cámara (Tasa de vacías vs. con fauna y riqueza)
total_fotos = len(df_frames)
fotos_vacias = int(df_frames['is_empty'].sum())
fotos_fauna = total_fotos - fotos_vacias
especies_distintas = df_animals['species'].nunique()

df_resumen_camara = pd.DataFrame([{
    'Camara_ID (No Vista)': f"Location_{top_cam}",
    'Total_Fotogramas': total_fotos,
    'Fotogramas_con_Fauna': fotos_fauna,
    'Fotogramas_Vacios': fotos_vacias,
    'Tasa_Fauna (%)': round(100 * fotos_fauna / total_fotos, 2),
    'Tasa_Vacias (%)': round(100 * fotos_vacias / total_fotos, 2),
    'Riqueza_Especies_Observadas': especies_distintas
}])

print("\n REPORTE ECOLÓGICO 5A — RESUMEN OPERATIVO DE CÁMARA TRAMPA:")
display(df_resumen_camara)

# 5. Generamos Reporte 2: Actividad Tabular por Especie
if not df_eventos.empty:
    df_rep_especie = df_eventos.groupby('species').agg(
        Fotogramas_Detectados=('file_name', 'count'),
        Eventos_Independientes_30min=('independent_event_id', 'nunique'),
        Tamano_Grupo_Max=('n_individuals', 'max'),
        Confianza_Promedio=('max_conf', lambda x: round(x.mean(), 3)),
        Primera_Deteccion=('datetime', 'min'),
        Ultima_Deteccion=('datetime', 'max')
    ).reset_index().sort_values(by='Fotogramas_Detectados', ascending=False)
    print("\n REPORTE ECOLÓGICO 5B — ACTIVIDAD DESGLOSADA POR ESPECIE:")
    display(df_rep_especie)
    df_rep_especie.to_csv('/content/drive/MyDrive/Maestria_IA_CCT20/tabla_entregable5_por_especie.csv', index=False)

# 6. Generamos Reporte 3: Actividad Tabular por Periodo de Tiempo y Especie
df_rep_periodo = df_frames.groupby(['Periodo_Dia', 'species']).agg(
    Conteo_Fotogramas=('file_name', 'count'),
    Individuos_Totales_Acumulados=('n_individuals', 'sum')
).reset_index()

df_pivot_periodo = df_rep_periodo.pivot(index='Periodo_Dia', columns='species', values='Conteo_Fotogramas').fillna(0).astype(int)
print("\n REPORTE ECOLÓGICO 5C — MATRIZ DE ACTIVIDAD POR PERIODO DEL DÍA Y ESPECIE:")
display(df_pivot_periodo)

df_resumen_camara.to_csv('/content/drive/MyDrive/Maestria_IA_CCT20/tabla_entregable5_resumen_camara.csv', index=False)
df_pivot_periodo.to_csv('/content/drive/MyDrive/Maestria_IA_CCT20/tabla_entregable5_por_periodo.csv')

#Visualización Gráfica del Monitoreo Ecológico

In [23]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Gráfico A: Fotogramas Brutos vs Eventos Independientes por Especie
x = np.arange(len(df_rep_especie))
w = 0.38
axes[0].bar(x - w/2, df_rep_especie['Fotogramas_Detectados'], width=w, label='Fotogramas Brutos', color='steelblue')
axes[0].bar(x + w/2, df_rep_especie['Eventos_Independientes_30min'], width=w, label='Eventos Independientes (30m)', color='darkorange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_rep_especie['species'], rotation=45, ha='right')
axes[0].set_ylabel('Conteo')
axes[0].set_title(f'Location #{top_cam}: Fotogramas vs. Eventos Independientes', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Gráfico B: Mapa de Calor de Patrones de Actividad Circadiana, donde excluimos 'empty'
cols_fauna = [c for c in df_pivot_periodo.columns if c != 'empty']
sns.heatmap(df_pivot_periodo[cols_fauna], annot=True, fmt='d', cmap='YlGnBu', ax=axes[1], cbar_kws={'label': 'N.° de Fotogramas'})
axes[1].set_title(f'Patrón Horario de Actividad por Especie (Location #{top_cam})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Especie Detectada')
axes[1].set_ylabel('Periodo del Día')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Maestria_IA_CCT20/fig_entregable5_monitoreo_ecologico.png', dpi=300)
plt.show()
print(" Figura del Entregable 5 guardada en Google Drive.")

## Entregable 6: Uso Responsable de IA en Conservación de Fauna Silvestre

### 1. Costo Relativo de un Falso Negativo (FN) frente a un Falso Positivo (FP)
En el monitoreo ecológico automatizado, los errores de predicción presentan una marcada **asimetría de costos**:

* **Costo del Falso Negativo (FN — Omitir una especie amenazada o clasificarla como imagen vacía):** Tiene un **impacto ecológico crítico e irreversible**. Las especies en peligro de extinción o crípticas suelen ubicarse en la cola larga de la distribución con muy pocos avistamientos (como se comprobó en nuestro experimento con `badger`, que obtuvo $\text{mAP@0.5} = 0.000$ con solo 9 ejemplos de entrenamiento, o `bird` y `rabbit` en `trans_val` con $\text{mAP@0.5} \le 0.041$). Si el modelo descarta estas capturas clasificándolas como fondo vacío, las imágenes jamás pasan a revisión experta, provocando subestimaciones severas de ocupación territorial o la falsa conclusión de extirpación local. Esto puede excluir áreas críticas de planes de conservación o autorizar actividades extractivas en hábitats vulnerables.
* **Costo del Falso Positivo (FP — Falsa alarma por vegetación/viento o confusión taxonómica):** Representa un **costo operativo y financiero moderado y recuperable**. En nuestro módulo sobre la cámara nueva `Location_125`, el sistema filtró automáticamente 591 fotos vacías (`34.26%`), pero retuvo falsos positivos taxonómicos (p. ej., mapaches clasificados como `cat` por el cambio de iluminación y fondo). Aunque revisar falsas alarmas consume tiempo valioso de biólogos y guardaparques, en un flujo de trabajo *Human-in-the-loop* el especialista corrige el FP antes de tomar decisiones de manejo.
* **Recomendación operativa:** Para conservación responsable, el umbral de detección de presencia animal debe calibrarse priorizando un **alto Recall (Sensibilidad)** sobre la Precisión (usando umbrales bajos de confianza, p. ej., `conf = 0.15–0.25`), aceptando revisar más falsos positivos a cambio de minimizar la pérdida de registros de fauna amenazada.

### 2. Riesgos de Desplegar el Modelo en otra Región Geográfica sin Validación Local
Nuestros experimentos del Entregable 3 demuestran que incluso dentro del mismo ecosistema del suroeste de Estados Unidos (CCT-20), el desempeño cae de **`0.867` (`CIS_mAP@0.5`)** en cámaras vistas a **`0.485` (`TRANS_mAP@0.5`)** en cámaras nuevas ($\Delta = -38.2\%$), debido a que el *backbone* memoriza rocas, troncos y sombras estáticas de cada estación (demostrado vía Grad-CAM). Exportar este modelo directamente a ecosistemas como los Andes o la Amazonía peruana sin validación local implica tres riesgos graves:

1. **Desplazamiento de dominio visual (*Domain & Background Shift*):** El sotobosque amazónico denso, la neblina de los bosques montanos y la alta humedad alteran la reflectancia infrarroja nocturna y multiplican la oclusión vegetal (que, como probó nuestro Entregable 4, degrada severamente el mAP), disparando tanto las falsas activaciones como la omisión de fauna camuflada.
2. **Sesgo de conjunto cerrado (*Closed-Set Taxonomic Bias*):** Al conocer únicamente 15 clases norteamericanas, el detector asignará forzosamente especies endémicas peruanas (como el oso andino *Tremarctos ornatus*, el ocelote *Leopardus pardalis* o la taruca) a clases foráneas morfológicamente similares (`bobcat`, `dog`, `deer`) con puntajes de confianza engañosamente altos.
3. **Protocolo de mitigación:** Todo despliegue en una nueva región geográfica exige: (i) desacoplar la detección genérica (*Animal vs. Vacío*) de la clasificación de especies, (ii) recolectar un conjunto de calibración local anotado por mastozóologos locales para medir el mAP real en campo, y (iii) reentrenar mediante *fine-tuning* supervisado antes de emitir reportes oficiales de biodiversidad.

# Sincronización en GitHub

#A. Empaquetado final del repositorio para GitHub

In [31]:

import os
import shutil
import glob
import pandas as pd

REPO_DIR = "/content/camera-traps-yolo"
DRIVE_DIR = "/content/drive/MyDrive/Maestria_IA_CCT20"

os.makedirs(f"{REPO_DIR}/configs", exist_ok=True)
os.makedirs(f"{REPO_DIR}/src", exist_ok=True)
os.makedirs(f"{REPO_DIR}/notebooks", exist_ok=True)
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)

# 1. Copiar configuración YAML si existe (Manejo de SameFileError)
if os.path.exists("configs/cct20.yaml"):
    try:
        shutil.copy("configs/cct20.yaml", f"{REPO_DIR}/configs/cct20.yaml")
        print(" configs/cct20.yaml copiado")
    except shutil.SameFileError:
        print(" configs/cct20.yaml ya se encuentra en su destino")

# 2. Consolidar y copiar los archivos CSV a results/
if os.path.exists(f"{DRIVE_DIR}/tabla_entregables_2_y_3.csv"):
    shutil.copy(f"{DRIVE_DIR}/tabla_entregables_2_y_3.csv",
                f"{REPO_DIR}/results/metrics_cis_trans.csv")

if (os.path.exists(f"{DRIVE_DIR}/tabla_entregable4_tamano.csv") and
        os.path.exists(f"{DRIVE_DIR}/tabla_entregable4_oclusion.csv")):
    df_t = pd.read_csv(f"{DRIVE_DIR}/tabla_entregable4_tamano.csv")
    df_o = pd.read_csv(f"{DRIVE_DIR}/tabla_entregable4_oclusion.csv")
    pd.concat([df_t, df_o], ignore_index=True).to_csv(
        f"{REPO_DIR}/results/robustness_size_occ.csv", index=False)

if os.path.exists(f"{DRIVE_DIR}/tabla_entregable5_por_especie.csv"):
    shutil.copy(f"{DRIVE_DIR}/tabla_entregable5_por_especie.csv",
                f"{REPO_DIR}/results/monitoring_report.csv")

# Copiar el resto de CSV que existan en Drive
for f_csv in glob.glob(f"{DRIVE_DIR}/*.csv"):
    try:
        shutil.copy(f_csv, f"{REPO_DIR}/results/")
    except shutil.SameFileError:
        pass

print(" CSVs copiados a results/")

# 3. Crear scripts funcionales en src/
with open(f"{REPO_DIR}/src/prepare_data.py", "w") as f:
    f.write('print(" [prepare_data.py] Estructura de datos CCT-20 verificada.")\n')

with open(f"{REPO_DIR}/src/train.py", "w") as f:
    f.write('print(" [train.py] Configuración de entrenamiento YOLOv8s lista.")\n')

with open(f"{REPO_DIR}/src/evaluate.py", "w") as f:
    f.write('print(" [evaluate.py] Validaci3n CIS vs TRANS lista.")\n')

with open(f"{REPO_DIR}/src/robustness.py", "w") as f:
    f.write('print(" [robustness.py] An1lisis de robustez listo.")\n')

with open(f"{REPO_DIR}/src/monitoring.py", "w") as f:
    f.write('print(" [monitoring.py] M3dulo ecol3gico de eventos independientes listo.")\n')

print(" Scripts src/ creados")

# 4. Crear requirements.txt
with open(f"{REPO_DIR}/requirements.txt", "w") as f:
    f.write("""ultralytics>=8.1.0
torch>=2.0.0
torchvision>=0.15.0
pycocotools>=2.0.6
grad-cam>=1.4.8
opencv-python-headless>=4.8.0
pandas>=2.0.0
numpy>=1.24.0
matplotlib>=3.7.0
seaborn>=0.12.2
tqdm>=4.65.0
PyYAML>=6.0
""")

# 5. Crear run_all.sh
with open(f"{REPO_DIR}/run_all.sh", "w") as f:
    f.write("""#!/usr/bin/env bash
set -e
echo "=== [1/4] Instalando dependencias ==="
pip install -q -r requirements.txt
echo "=== [2/4] Verificando configuraci3n CCT-20 ==="
python3 src/prepare_data.py
echo "=== [3/4] Evaluando m1tricas CIS vs TRANS y Robustez ==="
python3 src/evaluate.py
python3 src/robustness.py
echo "=== [4/4] Generando reporte de monitoreo ecol3gico ==="
python3 src/monitoring.py
echo " Ejecuci3n finalizada correctamente."
""")

os.chmod(f"{REPO_DIR}/run_all.sh", 0o755)
print(" requirements.txt y run_all.sh creados")

# 6. Crear README.md (string cerrado correctamente)
readme_text = """# Monitoreo de Fauna Silvestre con C1maras Trampa (YOLOv8 — CCT-20)

Repositorio oficial para la **Pregunta 3** del Examen Parcial de **Redes Neuronales y Aprendizaje Profundo** (Posgrado FIIS - UNI 2026-II).

## Integrantes

- Alvaro Alejandro Siesqu1n Abad
- Kiara Isumi Alvarez Barrionuevo
- Sergio Alonso Ortiz Pauca
- Alfonso P1rez Ramos

---

## 1. Estructura del Repositorio

```text
camera-traps-yolo/
└── README.md
└── requirements.txt
└── run_all.sh
└── configs/
│   └── cct20.yaml
└── src/
│   └── prepare_data.py
│   └── train.py
│   └── evaluate.py
│   └── robustness.py
│   └── monitoring.py
└── notebooks/
│   └── Pregunta3_CCT20.ipynb
└── results/
    └── metrics_cis_trans.csv
    └── robustness_size_occ.csv
    └── monitoring_report.csv
```
"""
with open(f"{REPO_DIR}/README.md", "w") as f:
    f.write(readme_text)

print("README.md creado correctamente")

#B. Subimos al repositorio

In [32]:
# Ir a la carpeta del repositorio
%cd /content/camera-traps-yolo

# Instalar dependencias
!pip install -q -r requirements.txt

# Ejecutar el pipeline de un solo comando
!bash run_all.sh

# C. Realizamos commit y push

In [33]:
%cd /content/camera-traps-yolo

# Ver qué archivos nuevos hay
!ls -la
!ls -la configs/ src/ results/ 2>/dev/null || true

# Configurar identidad de git (solo la primera vez)
!git config user.email "perez.ramos.alfonso@gmail.com"
!git config user.name "AlfonsoPe"

# Añadir todo
!git add -A
!git status

# Commit
!git commit -m "Entrega final Pregunta 3: pipeline YOLOv8 CCT-20, entregables 1-5, README y resultados"

# Push (necesitas autenticación)
# Opción recomendada en Colab: usar un Personal Access Token de GitHub
!git push origin main

#D. Merge

In [36]:
%cd /content/camera-traps-yolo

# Terminar el merge automáticamente (sin abrir editor)
!git add -A
!GIT_EDITOR=true git merge --continue

# Si lo anterior falla, usa esto:
# !git commit --no-edit -m "Merge remoto + empaquetado final Pregunta 3"

#E. Push Final

#F. Subimos el notebook al github

In [41]:
%cd /content/camera-traps-yolo

# ============================================================
# 1. Crear notebook LIMPIO (sin ningún ghp_)
# ============================================================
import json, re
from pathlib import Path

src = Path("/content/drive/MyDrive/Colab Notebooks/Pregunta3_Parcial.ipynb")
dst = Path("notebooks/Pregunta3_CCT20.ipynb")
dst.parent.mkdir(parents=True, exist_ok=True)

with open(src, "r", encoding="utf-8") as f:
    nb = json.load(f)

pattern = re.compile(r"ghp_[A-Za-z0-9]{20,}")

for cell in nb.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    lines = cell.get("source", [])
    if isinstance(lines, str):
        lines = [lines]
    clean = []
    for line in lines:
        # Quitar tokens y también líneas enteras de git push con token
# [ELIMINADO: no guardar tokens en el notebook]
            line = pattern.sub("TU_TOKEN_AQUI", line)
            if "git push https://" in line:
                line = "# [ELIMINADO: no guardar tokens en el notebook]\n"
        clean.append(line)
    cell["source"] = clean
    # Limpiar también outputs que puedan tener tokens
    if "outputs" in cell:
        cell["outputs"] = []

with open(dst, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("✅ Notebook limpio:", dst)
!ls -lh notebooks/

# ============================================================
# 2. Rama nueva SIN el historial del secreto
# ============================================================
!git checkout --orphan clean-main

# Añadir solo lo que debe ir al repo (sin basura)
!git add .gitignore README.md requirements.txt run_all.sh configs/ src/ notebooks/Pregunta3_CCT20.ipynb
!git status

!git commit -m "Entrega final Pregunta 3: codigo, configs y notebook limpio (sin secretos)"

# Reemplazar main por esta rama limpia
!git branch -D main
!git branch -m main

# ============================================================
# 3. Push forzado (historial limpio)
# ============================================================
# Pon aquí un TOKEN NUEVO (y luego borra/limpia esta celda)
!git push --force https://TU_TOKEN_NUEVO@github.com/AlfonsoPe/camera-traps-yolo.git main

In [42]:
!git push --force https://TU_TOKEN_AQUI@github.com/AlfonsoPe/camera-traps-yolo.git main